In [140]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import scipy.constants as pc
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.visualization import AsinhStretch, ImageNormalize, ManualInterval
import os

FILTERS =["F606W", "F814W","F115W", "F150W", "F200W", "F277W", "F356W", "F444W"]
# Individual filters increasing in wavelength
# Stacked filters are not useful for Sersíc fitting

CUTOUT_SIZES = ["0.96as", "3.0as"]
# The side lenghts of the plot out squares (each pixel is 0.03 as across) 

BROAD_LINE_DATA_PATH = "/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits"



In [141]:
def open_table(file_path):
        # Open the FITS file
        with fits.open(file_path) as hdul:
            # Show the HDU structure
            hdul.info()

            # Usually the table is in extension 1
            data = hdul[1].data

        # Convert to an Astropy Table
        tbl = Table(data)

        """
        print("\nColumns:")
        print(tbl.colnames)
        """
        return tbl

In [142]:
def cutout_plot(BROAD_LINE_DATA_PATH):

    table = open_table(BROAD_LINE_DATA_PATH)
    # All my raw data on the galaxies
    
    ids = table["SURVEY_ID"]
    surveys = table["SURVEY"]

    for i in range(len(ids)):

        id = ids[i]
        survey = surveys[i]
        # Create an output directory for this galaxy
        output_dir = f"/nvme/work/scratch/alberttg/Summer_project/Cutouts/{id}"
        os.makedirs(output_dir, exist_ok=True)  

        for filt in FILTERS:

            if survey in ["CEERSP1", "CEERSP2", "CEERSP3", "CEERSP4", "CEERSP5", "CEERSP6", "CEERSP7", "CEERSP7", "CEERSP8", "CEERSP9", "CEERSP10"]:

                    path = f"/raid/scratch/work/austind/GALFIND_WORK/Cutouts/v14/{survey}/ACS_WFC+NIRCam/0.96as/{filt}/SExtractor2.25.0_rms_F277W+F356W+F444W_(0.32)as/data/{id}.fits"

            if survey in ["PRIMER-COSMOS", "PRIMER-UDS"]:

                    path = f"/raid/scratch/work/austind/GALFIND_WORK/Cutouts/v12_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.96as/{filt}/data_native/{id}.fits"

            if survey in ["JADES-DR3-GS-East", "JADES-DR3-GS-North", "JADES-DR3-GS-South", "JADES-DR3-GS-West", "JADES-DR3-GN-Parallel", "JADES-DR3-GN-PMedium", "JADES-DR3-GN-Deep"]:

                    path = f"/raid/scratch/work/austind/GALFIND_WORK/Cutouts/v13/{survey}/ACS_WFC+NIRCam/0.96as/{filt}/data_native/{id}.fits"

            else:
                    print(f"Galaxy {id} could not be found in these surveys")
                    break

            fig, ax = plt.subplots(figsize=(6,6))
            # open fits file and plot the image
            hdul = fits.open(path)
            data = hdul[1].data

            vmin, vmax = np.percentile(data, (5,99))
            # Maybe change for visualisation

            norm = ImageNormalize(data, interval=ManualInterval(vmin=vmin, vmax=vmax), # automatic contrast (like DS9)
                                  stretch=AsinhStretch())                      
            # Asinh keeps faint structures but becomes log at high flux to show structural differences at centre
            ax.imshow(data, norm=norm, origin="lower", cmap="gray")
            
            # label filt in upper right
            # (0,0) is bottom left
            # (1,1) is top right
            ax.text(0.98, 0.98, f"{filt} {id}", transform=ax.transAxes, ha="right", va="top", fontsize=12, bbox=dict(boxstyle="round,pad=0.3",facecolor="white",edgecolor="black",alpha=0.8))

            ax.set_xticks([])
            ax.set_yticks([])
            # Removes tick marks
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"{filt}.png"), dpi=300, bbox_inches="tight")
            plt.close(fig)

In [143]:
if __name__ == "__main__":

    cutout_plot(BROAD_LINE_DATA_PATH)

Filename: /nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     54   144R x 23C   [D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, 18A, 21A, J]   


PermissionError: [Errno 13] Permission denied: '/nvme/work'